# Exercise 3: Modeling Data for OLAP Use Cases

We've seen how data structures can be adjusted for optimal writing and updating. Even for retreiving individual records. However, there are use cases where we are primarily concerned with reading a lot of data at once, aggregating it, and analyzing it. These, recall from our first lesson - are OLAP or Online Analytical Processing use cases. Such a use case requires a slightly different data model for optimal reading. 

## Step 1: Preparing Tables & Data
Our journey will have two parts: first, we'll define the Star Schema for our use case. Then we'll refine the Star Schema into a Snowflake Schema by further normalizing a dimension.
Lastly, we'll query the Snowflake Schema by joining all the tables back together to build a final, "denormalized" report for an end-user. This simulates a common workflow: starting with a good structure, and then building the reporting queries needed for analysis.

Let's create our Star Schema. This schema is already normalized and consists of one fact table and three dimension tables.

Notice that DimProduct still contains the product_category as a text field. This is a common feature of a simple Star Schema.

(Run this code block to set up the data)

In [5]:
%%capture
%pip install notebook duckdb==1.3 jupysql matplotlib duckdb-engine psycopg2;
%config SqlMagic.autopandas = True;
%config SqlMagic.feedback = False;
%config SqlMagic.displaycon = False;
import duckdb;
import pandas as pd
import zipfile;
import os;
from io import StringIO
import psycopg2
from psycopg2 import Error

def connect():
    # 1. Establish the connection
    try:
        # Connect to the PostgreSQL database
        conn = psycopg2.connect(
            dbname='postgres',
            user='postgres',
            password='I4NAennDnRh1uO9z2y0k6pMfAf',
            host='localhost',
            port='5432')

        cursor = conn.cursor()
        print("Connection successful.")
        return cursor, conn
    
    except psycopg2.Error as e:
        # Handle any errors during connection or query execution
        print(f"\nAn error occurred: {e}")

def execute_query(cursor, conn, sql_query, fetch = False, commit = True):
    cursor.execute(sql_query)
    if fetch == True:
        results = cursor.fetchall()
        return results
    if commit == True:
        conn.commit()

In [6]:
sql_query = """
BEGIN;
-- Create the Dimension Tables for our Star Schema
CREATE TABLE DimCustomer (
    customer_key SERIAL PRIMARY KEY,
    customer_id INT,
    customer_name VARCHAR(100),
    customer_state VARCHAR(50)
);

CREATE TABLE DimProduct (
    product_key SERIAL PRIMARY KEY,
    product_id INT,
    product_name VARCHAR(100),
    product_category VARCHAR(50), -- Note: This is part of the dimension
    unit_price DECIMAL(10, 2)
);

CREATE TABLE DimDate (
    date_key SERIAL PRIMARY KEY,
    full_date DATE NOT NULL,
    month INT,
    year INT
);

-- Populate the Dimension Tables
INSERT INTO DimCustomer (customer_id, customer_name, customer_state) VALUES
(101, 'Alice Smith', 'CA'),
(102, 'Bob Johnson', 'NY'),
(103, 'Charlie Day', 'PA');

INSERT INTO DimProduct (product_id, product_name, product_category, unit_price) VALUES
(1, 'Laptop', 'Electronics', 1200.00),
(2, 'Mouse', 'Electronics', 25.00), -- 'Electronics' is repeated
(3, 'Desk Chair', 'Furniture', 150.00);

INSERT INTO DimDate (full_date, month, year) VALUES
('2024-01-15', 1, 2024),
('2024-01-16', 1, 2024),
('2024-01-17', 1, 2024),
('2024-01-18', 1, 2024);

-- Create the Fact Table
CREATE TABLE FactSales (
    sales_key SERIAL PRIMARY KEY,
    date_key INT REFERENCES DimDate(date_key),
    customer_key INT REFERENCES DimCustomer(customer_key),
    product_key INT REFERENCES DimProduct(product_key),
    quantity INT,
    total_sale_amount DECIMAL(10, 2)
);

-- Populate the Fact Table (using the keys from the dimensions)
INSERT INTO FactSales (date_key, customer_key, product_key, quantity, total_sale_amount) VALUES
-- (Order 1: Alice, Laptop)
(1, 1, 1, 1, 1200.00),
-- (Order 2: Bob, Mouse)
(2, 2, 2, 2, 50.00),
-- (Order 3: Alice, Mouse)
(2, 1, 2, 1, 25.00),
-- (Order 4: Charlie, Desk Chair)
(3, 3, 3, 1, 150.00),
-- (Order 5: Bob, Laptop)
(4, 2, 1, 1, 1200.00);
COMMIT;
"""
connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = False)

Connection successful.


DuplicateTable: relation "dimcustomer" already exists


## Check your starting tables!

In [14]:
sql_query = "SELECT * FROM DimCustomer;"
connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = True)

Connection successful.


[(1, 101, 'Alice Smith', 'CA'),
 (2, 102, 'Bob Johnson', 'NY'),
 (3, 103, 'Charlie Day', 'PA')]

In [18]:
sql_query = "SELECT * FROM DimProduct;"
connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = True)

Connection successful.


[(1, 1, 'Laptop', 'Electronics', Decimal('1200.00')),
 (2, 2, 'Mouse', 'Electronics', Decimal('25.00')),
 (3, 3, 'Desk Chair', 'Furniture', Decimal('150.00'))]

In [19]:
sql_query = "SELECT * FROM DimDate;"
connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = True)

Connection successful.


[(1, datetime.date(2024, 1, 15), 1, 2024),
 (2, datetime.date(2024, 1, 16), 1, 2024),
 (3, datetime.date(2024, 1, 17), 1, 2024),
 (4, datetime.date(2024, 1, 18), 1, 2024)]

In [20]:
sql_query =" SELECT * FROM FactSales;"
connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = True)

Connection successful.


[(1, 1, 1, 1, 1, Decimal('1200.00')),
 (2, 2, 2, 2, 2, Decimal('50.00')),
 (3, 2, 1, 2, 1, Decimal('25.00')),
 (4, 3, 3, 3, 1, Decimal('150.00')),
 (5, 4, 2, 1, 1, Decimal('1200.00'))]

Next, let's refine to a Snowflake Schema
Our Star Schema is good, but DimProduct still has some redundancy. The category "Electronics" is repeated. This is a transitive dependency (product_name -> product_category), which we can normalize further. A Snowflake Schema fixes this by normalizing the dimensions themselves. We'll create a new DimCategory table and link DimProduct to it.

A. Create and Populate DimCategory
(Fill in the /* ??? */ blanks)

In [28]:
sql_query = """
BEGIN;
/*
 * 1. Create the new dimension
 * This table will hold the unique category names.
 */
CREATE TABLE DimCategory (
    category_key SERIAL PRIMARY KEY,
     /* ??? -- What is the one attribute for this table? (e.g., 'Electronics', 'Furniture') */
    category_name VARCHAR(50) UNIQUE
);

/*
 * 2. Populate it with unique values from the DimProduct table
 */
INSERT INTO DimCategory (category_name)
SELECT DISTINCT
    /* ??? -- Which column from DimProduct holds the category? */
    product_category
FROM DimProduct;

-- Check your work
SELECT * FROM DimCategory;
COMMIT;
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = False)

Connection successful.


DuplicateTable: relation "dimcategory" already exists


Now to "Snowflake" the DimProduct Table. We'll modify DimProduct to use our new DimCategory table. This involves three steps:

Add a new foreign key column (category_key).

Update that column by joining with DimCategory.

Drop the old, redundant text column (product_category).

(Fill in the /* ??? */ blanks)

In [30]:
sql_query = """
BEGIN;
/*
 * 1. Add the new foreign key column to DimProduct
 */
ALTER TABLE DimProduct
ADD COLUMN /* ??? -- The new FK column name */
    category_key INT REFERENCES DimCategory(category_key);

/*
 * 2. Update the table to set the new FK values
 * We join DimProduct with DimCategory to find the right key
 */
UPDATE DimProduct
SET category_key = dc.category_key
FROM DimCategory AS dc
WHERE DimProduct.product_category = /* ??? -- The join condition between the two tables */
    dc.category_name;

/*
 * 3. Drop the old redundant column
 */
ALTER TABLE DimProduct
DROP COLUMN product_category; /* ??? -- The old text column */;
COMMIT;
"""

connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = False)

Connection successful.


In [31]:
sql_query = """
-- Check your final, "snowflaked" table!
-- Notice the 'product_category' column is gone, replaced by 'category_key'
SELECT * FROM DimProduct;
"""
connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = True)

Connection successful.


[(1, 1, 'Laptop', Decimal('1200.00'), 2),
 (2, 2, 'Mouse', Decimal('25.00'), 2),
 (3, 3, 'Desk Chair', Decimal('150.00'), 1)]

The final step: let's build the Final Report (Denormalization)
You have successfully created a Snowflake Schema!

Now, for our final task, management wants a simple report. They don't want to see customer_key or product_key; they want names and descriptions. Our job is to write a single query that joins all our normalized tables back together to create one big, flat report.

This process of joining from a schema to create a flat report is sometimes called denormalization for reading.

A. Write the Report Query
(Fill in the /* ??? */ blanks to join all the tables)

Hint: You will need to join 5 tables in total:

Start FROM the FactSales table.

JOIN it to DimDate.

JOIN it to DimCustomer.

JOIN it to DimProduct.

JOIN DimProduct to DimCategory (the "snowflake" join).

In [34]:
sql_query = """
SELECT
    d.full_date,
    c.customer_name,
    c.customer_state,
    p.product_name,
    cat.category_name,
    f.quantity,
    p.unit_price,
    f.total_sale_amount
FROM
    FactSales AS f /* ??? -- Start with the central table */
JOIN
    DimDate AS d ON /* ??? -- Join facts to dates */
    f.date_key = d.date_key
JOIN
    DimCustomer AS c ON /* ??? -- Join facts to customers */
    f.customer_key = c.customer_key
JOIN
    DimProduct AS p ON /* ??? -- Join facts to products */
    f.product_key = p.product_key
JOIN
    DimCategory AS cat ON /* ??? -- This is the snowflake join (product to category) */
    p.category_key = cat.category_key
ORDER BY
    d.full_date;
"""
connection = connect()
execute_query(connection[0], connection[1], sql_query, fetch = True)

Connection successful.


[(datetime.date(2024, 1, 15),
  'Alice Smith',
  'CA',
  'Laptop',
  'Electronics',
  1,
  Decimal('1200.00'),
  Decimal('1200.00')),
 (datetime.date(2024, 1, 16),
  'Bob Johnson',
  'NY',
  'Mouse',
  'Electronics',
  2,
  Decimal('25.00'),
  Decimal('50.00')),
 (datetime.date(2024, 1, 16),
  'Alice Smith',
  'CA',
  'Mouse',
  'Electronics',
  1,
  Decimal('25.00'),
  Decimal('25.00')),
 (datetime.date(2024, 1, 17),
  'Charlie Day',
  'PA',
  'Desk Chair',
  'Furniture',
  1,
  Decimal('150.00'),
  Decimal('150.00')),
 (datetime.date(2024, 1, 18),
  'Bob Johnson',
  'NY',
  'Laptop',
  'Electronics',
  1,
  Decimal('1200.00'),
  Decimal('1200.00'))]

B. Solution: Run the Final Query
Run the query below (which is the solution to the exercise above) to see your final report. Notice how this report looks very similar to the "bad" flat file we might have started with, but it's generated on-the-fly from a clean, fast, and normalized set of tables.

(This is the "answer key" query. Just run it.)

In [32]:
SELECT
    d.full_date,
    c.customer_name,
    c.customer_state,
    p.product_name,
    cat.category_name,
    f.quantity,
    p.unit_price,
    f.total_sale_amount
FROM
    FactSales AS f
JOIN
    DimDate AS d ON f.date_key = d.date_key
JOIN
    DimCustomer AS c ON f.customer_key = c.customer_key
JOIN
    DimProduct AS p ON f.product_key = p.product_key
JOIN
    DimCategory AS cat ON p.category_key = cat.category_key
ORDER BY
    d.full_date;

IndentationError: unexpected indent (1429963294.py, line 2)